# 02 — Contribution 1: Semantic embedding study

This notebook evaluates the alternative semantic embeddings on the DARPA TRACE test set, keeping the rest of the STGAN architecture identical.

## Configurations evaluated
- BERT (sentence-transformers MiniLM-L6-v2, 384-dim)
- EmbeddingGemma raw (google/embeddinggemma-300m, 768-dim)
- EmbeddingGemma prompted (task-aware prompt)

The Word2Vec baseline is reported in `01_baseline_reproduction.ipynb` for comparison.

## Expected results
| Embedding | F1 |
|---|---|
| Word2Vec (baseline) | ≈ 0.9875 |
|FastText|≈ 0.9725|
| BERT | ≈ 0.9769 |
| EmbeddingGemma raw | ≈ 0.9408 |
| EmbeddingGemma prompted | ≈ 0.9738 |

## Prerequisites
- `01_baseline_reproduction.ipynb` completed.
- HuggingFace token with access to `google/embeddinggemma-300m`, set as the `HF_TOKEN` environment variable.
- Trained weights present under `trained_weights/trace/{c1_bert, c1_gemma, c1_gemma_prompted}/`.

## Reproducibility note
Each eval cell will regenerate the BERT/Gemma embedding caches if absent (~25 min per cache). The caches consume ~3 GB each. They are excluded from the depot and regenerated automatically.

## Next
`03_C2_architectures.ipynb`.

In [4]:
print("=== Notebook 02 — Contribution 1: Semantic embedding study ===")

=== Notebook 02 — Contribution 1: Semantic embedding study ===


In [6]:
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("STGAN_ROOT", Path.cwd().resolve().parent))
assert (PROJECT_ROOT / "code" / "eval").exists(), f"PROJECT_ROOT incorrect : {PROJECT_ROOT}"

# Pour le reproducteur : set HF_TOKEN via les settings HuggingFace de ton compte
HF_TOKEN = os.environ.get("HF_TOKEN", "<YOUR_HF_TOKEN>")
if HF_TOKEN == "<YOUR_HF_TOKEN>":
    print("⚠ HF_TOKEN non configuré — Gemma eval va échouer")
else:
    print("✓ HF_TOKEN configuré")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

✓ HF_TOKEN configuré
PROJECT_ROOT = /home/jovyan/stgan/work/STGAN-master


In [7]:
import subprocess, time

script = f"""
cd {PROJECT_ROOT}/code/eval
PYTHONUNBUFFERED=1 PYTHONPATH={PROJECT_ROOT}/code \\
nohup python -u eval_trace_bert.py > /tmp/eval_bert.log 2>&1 &
echo $! > /tmp/eval_bert.pid
"""
subprocess.run(["bash", "-c", script], check=True)
time.sleep(3)
print(f"✓ BERT eval lancé (PID {open('/tmp/eval_bert.pid').read().strip()})")
print("Temps : ~25 min (pre-encoding BERT + eval)")

✓ BERT eval lancé (PID 21210)
Temps : ~25 min (pre-encoding BERT + eval)


In [9]:
!tail -10 /tmp/eval_bert.log

Epoch: 45                                            
Epoch: 46                                            
Epoch: 47                                            
Epoch: 48                                            
Epoch: 49                                            
[TIMING] infer_seconds=193.2                         
True Positives: 67381, False Positives: 2391, False Negatives: 792,True Negatives: 1136423
Precision: 0.9657, Recall: 0.9884, Fscore: 0.9769
Accuracy: 0.9974
[RESUME] emb=bert gnn=gat fusion=mha


In [10]:
import subprocess, time

script = f"""
cd {PROJECT_ROOT}/code/eval
PYTHONUNBUFFERED=1 PYTHONPATH={PROJECT_ROOT}/code \\
nohup python -u eval_trace_c2.py --gnn gat --fusion mha > /tmp/eval_fasttext.log 2>&1 &
echo $! > /tmp/eval_fasttext.pid
"""
subprocess.run(["bash", "-c", script], check=True)
time.sleep(3)
print(f"✓ FastText eval lancé (PID {open('/tmp/eval_fasttext.pid').read().strip()})")
print("Temps : ~10 min (eval seul, FastText déjà entraîné)")

✓ FastText eval lancé (PID 21370)
Temps : ~10 min (eval seul, FastText déjà entraîné)


In [11]:
import os, time
pid = open("/tmp/eval_fasttext.pid").read().strip()
while os.path.exists(f"/proc/{pid}"):
    time.sleep(30); print(".", end="", flush=True)
print("\n✓ FastText eval terminé\n")
!tail -10 /tmp/eval_fasttext.log

.................
✓ FastText eval terminé

Epoch: 45                                            
Epoch: 46                                            
Epoch: 47                                            
Epoch: 48                                            
Epoch: 49                                            
[TIMING] infer_seconds=267.4                         
True Positives: 67381, False Positives: 3015, False Negatives: 792,True Negatives: 1137060
Precision: 0.9572, Recall: 0.9884, Fscore: 0.9725
Accuracy: 0.9968
[RESUME] gnn=gat fusion=mha params=452894


In [12]:
import os, time
pid = open("/tmp/eval_bert.pid").read().strip()
while os.path.exists(f"/proc/{pid}"):
    time.sleep(30); print(".", end="", flush=True)
print("\n✓ BERT eval terminé\n")
!tail -10 /tmp/eval_bert.log


✓ BERT eval terminé

Epoch: 45                                            
Epoch: 46                                            
Epoch: 47                                            
Epoch: 48                                            
Epoch: 49                                            
[TIMING] infer_seconds=193.2                         
True Positives: 67381, False Positives: 2391, False Negatives: 792,True Negatives: 1136423
Precision: 0.9657, Recall: 0.9884, Fscore: 0.9769
Accuracy: 0.9974
[RESUME] emb=bert gnn=gat fusion=mha


In [15]:
import subprocess, time

script = f"""
cd {PROJECT_ROOT}/code/eval
HF_TOKEN={HF_TOKEN} \\
PYTORCH_NVML_BASED_CUDA_CHECK=0 \\
PYTHONUNBUFFERED=1 PYTHONPATH={PROJECT_ROOT}/code \\
nohup python -u eval_trace_gemma.py --variant raw > /tmp/eval_gemma_raw.log 2>&1 &
echo $! > /tmp/eval_gemma_raw.pid
"""
subprocess.run(["bash", "-c", script], check=True)
time.sleep(3)
print(f"✓ Gemma raw eval lancé (PID {open('/tmp/eval_gemma_raw.pid').read().strip()})")
print("Temps : ~40 min (pre-encoding Gemma + eval)")

✓ Gemma raw eval lancé (PID 21510)
Temps : ~40 min (pre-encoding Gemma + eval)


In [16]:
import os, time
pid = open("/tmp/eval_gemma_raw.pid").read().strip()
while os.path.exists(f"/proc/{pid}"):
    time.sleep(30); print(".", end="", flush=True)
print("\n✓ Gemma raw eval terminé\n")
!tail -10 /tmp/eval_gemma_raw.log

...............................................................................................................
✓ Gemma raw eval terminé

Epoch: 45                                            
Epoch: 46                                            
Epoch: 47                                            
Epoch: 48                                            
Epoch: 49                                            
[TIMING] infer_seconds=189.4                         
True Positives: 67383, False Positives: 7685, False Negatives: 790,True Negatives: 1130319
Precision: 0.8976, Recall: 0.9884, Fscore: 0.9408
Accuracy: 0.993
[RESUME] emb=gemma_raw gnn=gat fusion=mha


In [19]:
import subprocess, time

script = f"""
cd {PROJECT_ROOT}/code/eval
HF_TOKEN={HF_TOKEN} \\
PYTORCH_NVML_BASED_CUDA_CHECK=0 \\
PYTHONUNBUFFERED=1 PYTHONPATH={PROJECT_ROOT}/code \\
nohup python -u eval_trace_gemma.py --variant prompted > /tmp/eval_gemma_prompted.log 2>&1 &
echo $! > /tmp/eval_gemma_prompted.pid
"""
subprocess.run(["bash", "-c", script], check=True)
time.sleep(3)
print(f"✓ Gemma prompted eval lancé (PID {open('/tmp/eval_gemma_prompted.pid').read().strip()})")
print("Temps : ~40 min")

✓ Gemma prompted eval lancé (PID 21648)
Temps : ~40 min


In [21]:
import os, time
pid = open("/tmp/eval_gemma_prompted.pid").read().strip()
while os.path.exists(f"/proc/{pid}"):
    time.sleep(30); print(".", end="", flush=True)
print("\n✓ Gemma prompted eval terminé\n")
!tail -10 /tmp/eval_gemma_prompted.log


✓ Gemma prompted eval terminé

Epoch: 45                                            
Epoch: 46                                            
Epoch: 47                                            
Epoch: 48                                            
Epoch: 49                                            
[TIMING] infer_seconds=187.0                         
True Positives: 67382, False Positives: 2834, False Negatives: 791,True Negatives: 1134733
Precision: 0.9596, Recall: 0.9884, Fscore: 0.9738
Accuracy: 0.997
[RESUME] emb=gemma_prompted gnn=gat fusion=mha
